# Melhoria da detecção de valores anômalos

Rodada independente de calibração e avaliação final, com resultados anteriores preservados.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

root = Path.cwd()
while not (root / "ml/config.py").exists():
    if root.parent == root:
        raise RuntimeError("Raiz do projeto não encontrada")
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

## Nova rodada de melhoria com teste reservado

Esta etapa foi planejada após observar os limites anteriores. Os experimentos antigos permanecem preservados. A seleção e a calibração usam novos clientes de desenvolvimento, e a avaliação final usa cinco novas sementes em maio de 2026. O lote antigo aparece apenas como diagnóstico posterior.

In [2]:
from ml.improvement import DEST as IMPROVEMENT_RESULTS
from ml.improvement_report import make_figures as improvement_figures

improvement_summary = json.loads(
    (IMPROVEMENT_RESULTS / "resumo.json").read_text(encoding="utf-8")
)
display(
    Markdown(
        (IMPROVEMENT_RESULTS / "RESULTADOS_MELHORIA.md").read_text(encoding="utf-8")
    )
)
display(pd.read_csv(IMPROVEMENT_RESULTS / "validacao.csv").round(4))
display(pd.read_csv(IMPROVEMENT_RESULTS / "teste_agregado.csv").round(4))
display(pd.read_csv(IMPROVEMENT_RESULTS / "teste_por_cliente.csv").round(4))
improvement_plots = improvement_figures()
if "show_figures" in globals():
    show_figures(improvement_plots, "6.5. Nova rodada sem documentos")
else:
    for figure in improvement_plots:
        display(figure)
        plt.close(figure)
if "required" in globals():
    required.extend(
        IMPROVEMENT_RESULTS / name
        for name in [
            "protocolo.json",
            "selecao_congelada.json",
            "resumo.json",
            "validacao.csv",
            "teste_agregado.csv",
            "RESULTADOS_MELHORIA.md",
        ]
    )

# Melhoria do detector com novo teste reservado

## Protocolo e separação

Os resultados antigos foram preservados. A rodada foi desenhada após inspecionar a simulação anterior. Não é uma confirmação cega de toda a pesquisa. A configuração foi registrada localmente antes da execução, em `protocolo.json`.

- Desenvolvimento: três clientes sintéticos, sementes 81001, 81002 e 81003. Históricos de janeiro a março de 2026 para ajuste não supervisionado; lotes de abril para calibração e escolha.
- Teste: cinco novos clientes, sementes 91001 a 91005, com histórico de janeiro a abril e lotes de maio de 2026. Nenhuma seleção usou estes resultados.
- Mesmas regras do gerador de cliente já conhecido, com novas realizações. A novidade dos registros não representa outra fonte empresarial nem garante generalização a outros geradores.
- Candidatos: original congelado, histórico recalibrado, medianas recentes em 20 e 60 dias e Isolation Forest contextual. O Isolation Forest usa dez atributos de desvios, dispersão, frequência e deslocamento recente, com padronização ajustada somente nos históricos de desenvolvimento, sem rótulos.
- Medianas e dispersões usam só dias anteriores. Escala: máximo entre 1,4826 vezes o desvio absoluto mediano e 0,10. Menos de cinco observações na janela leva à abstenção da regra, com score zero. Nenhuma ausência de alerta comprova normalidade.
- Seleção por maior F0,5 na validação, com recall mínimo de 35% e pelo menos dez alertas. Esse piso é experimental, não foi definido por uma área de negócio. Não há garantia de que se mantenha em cada novo cliente.
- Comparação final predefinida: precisão e F1 maiores que o original, sem perda de recall superior a dez pontos percentuais no teste agregado. Não houve alteração de candidatos após examinar o teste.

## Resultado no teste reservado

O selecionado foi **Mediana 60 dias**. Nos cinco lotes, há 7555 transações e 310 adulterações. A precisão passou de **15.77% para 24.13%**; o recall, de **50.00% para 31.29%**; e o F1, de **0.2398 para 0.2725**. A AP foi 0.2361 no original e 0.2168 no selecionado.

O original emitiu 983 alertas, com 155 verdadeiros positivos e 828 falsos positivos. A nova configuração emitiu 402 alertas, com 97 verdadeiros positivos, 305 falsos positivos e 213 falsos negativos. O resultado **não atendeu** ao critério de melhoria registrado. Todos os candidatos, inclusive os não selecionados, estão em `teste_agregado.csv`.

A precisão por cliente do selecionado variou de 16.22% a 30.00%. A métrica agregada junta as transações dos cinco lotes, não é a média simples das precisões. A comparação é pareada dentro de cada lote. Não se atribui significância estatística ou intervalo populacional a apenas cinco realizações sintéticas.

## Lote antigo como diagnóstico

No lote já conhecido, a nova configuração obteve precisão de 23.08%, recall de 31.34% e F1 de 0.2658. Esse resultado é retrospectivo e fica separado do teste reservado. Os 14,85% anteriores continuam documentados como resultado da configuração original.

## Interpretação e limites

Precisão maior precisa ser interpretada junto ao recall e ao volume de alertas. O detector continua recebendo somente valores e contexto de transação. Um pico legítimo e uma adulteração podem ser observacionalmente semelhantes. Nenhum dado de autorização ou explicação operacional foi inventado para facilitar a tarefa. Não foi demonstrada prontidão para produção ou prevenção de fraude.

Uma nova calibração em dados que se aproximam do ambiente de aplicação pode mudar bastante o ponto de operação. A comparação separa o ganho da decisão de limiar do ganho da representação: a linha de histórico recalibrado preserva exatamente o score antigo. O vencedor é escolhido na validação e não substituído por outro candidato que eventualmente pareça melhor no teste.

## Reprodução

Python 3.11/3.12 e as versões declaradas em `ml/requirements.txt`. Executar `python -m ml.improvement` a partir da raiz. O protocolo inclui versões, sementes e hashes do código e dos 33 arquivos originais. Datasets, modelos e previsões ficam em `ml/artifacts/melhoria/rodada_02` e `ml/reports/melhoria/rodada_02`. O notebook `ml/07_melhoria_modelo.ipynb` apenas lê os resultados, sem treinar ou reajustar.


,modelo,average_precision,roc_auc,precision,recall,f1,precision_top_5pct,recall_top_5pct,alertas,taxa_alerta,tn,fp,fn,tp,limiar,n,anomalias,prevalencia,f0_5
0,Mediana 60 dias,0.1933,0.7145,0.2509,0.350,0.2923,0.2845,0.340,279,0.0584,4368,209,130,70,2.7882,4777,200,0.0419,0.2660
1,Historico recalibrado,0.2028,0.7150,0.2457,0.355,0.2904,0.2594,0.310,289,0.0605,4359,218,129,71,2.2539,4777,200,0.0419,0.2618
2,Mediana 20 dias,0.1901,0.6702,0.2128,0.350,0.2647,0.2343,0.280,329,0.0689,4318,259,130,70,2.8702,4777,200,0.0419,0.2309
3,Isolation Forest contextual,0.1752,0.7000,0.1854,0.355,0.2436,0.2469,0.295,383,0.0802,4265,312,129,71,0.5443,4777,200,0.0419,0.2050
4,Original congelado,0.2028,0.7150,0.1465,0.455,0.2217,0.2594,0.310,621,0.1300,4047,530,109,91,1.6812,4777,200,0.0419,0.1695


,modelo,selecionado,average_precision,roc_auc,precision,recall,f1,precision_top_5pct,recall_top_5pct,alertas,taxa_alerta,tn,fp,fn,tp,limiar,n,anomalias,prevalencia,f0_5
0,Historico recalibrado,False,0.2361,0.7291,0.2510,0.3871,0.3046,0.2831,0.3452,478,0.0633,6887,358,190,120,2.2539,7555,310,0.041,0.2700
1,Isolation Forest contextual,False,0.1971,0.6981,0.1897,0.3581,0.2480,0.2381,0.2903,585,0.0774,6771,474,199,111,0.5443,7555,310,0.041,0.2094
2,Mediana 20 dias,False,0.1821,0.6817,0.1954,0.3290,0.2452,0.2275,0.2774,522,0.0691,6825,420,208,102,2.8702,7555,310,0.041,0.2127
3,Mediana 60 dias,True,0.2168,0.7240,0.2413,0.3129,0.2725,0.2513,0.3065,402,0.0532,6940,305,213,97,2.7882,7555,310,0.041,0.2529
4,Original congelado,False,0.2361,0.7291,0.1577,0.5000,0.2398,0.2831,0.3452,983,0.1301,6417,828,155,155,1.6812,7555,310,0.041,0.1827


,seed,modelo,average_precision,roc_auc,precision,recall,f1,precision_top_5pct,recall_top_5pct,alertas,taxa_alerta,tn,fp,fn,tp,limiar,n,anomalias,prevalencia,f0_5
0,91001,Original congelado,0.1827,0.6963,0.1020,0.4444,0.1660,0.2025,0.3556,196,0.1248,1349,176,25,20,1.6812,1570,45,0.0287,0.1206
1,91001,Historico recalibrado,0.1827,0.6963,0.1979,0.4222,0.2695,0.2025,0.3556,96,0.0611,1448,77,26,19,2.2539,1570,45,0.0287,0.2214
2,91001,Mediana 20 dias,0.1310,0.6622,0.1389,0.3333,0.1961,0.1646,0.2889,108,0.0688,1432,93,30,15,2.8702,1570,45,0.0287,0.1572
3,91001,Mediana 60 dias,0.2214,0.6763,0.1622,0.2667,0.2017,0.1646,0.2889,74,0.0471,1463,62,33,12,2.7882,1570,45,0.0287,0.1760
4,91001,Isolation Forest contextual,0.1452,0.6686,0.1596,0.3333,0.2158,0.1899,0.3333,94,0.0599,1446,79,30,15,0.5443,1570,45,0.0287,0.1781
5,91002,Original congelado,0.2954,0.7703,0.1698,0.5714,0.2618,0.3562,0.4127,212,0.1461,1212,176,27,36,1.6812,1451,63,0.0434,0.1976
6,91002,Historico recalibrado,0.2954,0.7703,0.2959,0.4603,0.3602,0.3562,0.4127,98,0.0675,1319,69,34,29,2.2539,1451,63,0.0434,0.3187
7,91002,Mediana 20 dias,0.2143,0.7004,0.2178,0.3492,0.2683,0.2329,0.2698,101,0.0696,1309,79,41,22,2.8702,1451,63,0.0434,0.2355
8,91002,Mediana 60 dias,0.2460,0.7612,0.2500,0.3016,0.2734,0.2603,0.3016,76,0.0524,1331,57,44,19,2.7882,1451,63,0.0434,0.2589
9,91002,Isolation Forest contextual,0.2097,0.7223,0.1575,0.3651,0.2201,0.2740,0.3175,146,0.1006,1265,123,40,23,0.5443,1451,63,0.0434,0.1777


<Figure size 990x528 with 1 Axes>

<Figure size 990x528 with 1 Axes>

<Figure size 990x528 with 1 Axes>